## 0. channel_id 기준 JOIN

`all_channels`와 `all_videos`를 `channel_id`로 조인합니다.  
영상 파일은 채널당 여러 행이 있으므로 조인 방식에 따라 결과가 달라집니다.

| 방식 | 설명 | 용도 |
|---|---|---|
| **wide (채널 단위)** | videos를 채널 단위로 집계 후 1:1 조인 | 머신러닝 (XGBoost 등) |
| **long (영상 단위)** | 채널 정보를 각 영상 행에 붙이는 1:N 조인 | 시계열 분석, LSTM |

아래는 두 방식 모두 제공합니다.


### 1. Wide 조인 (채널 단위, ML용)

In [1]:
import pandas as pd
import os

In [2]:
DATA_DIR_channels = "/Users/sj.kang/Desktop/project2/SKN30-2nd-1Team/data/raw/channels/csv"
DATA_DIR_videos = "/Users/sj.kang/Desktop/project2/SKN30-2nd-1Team/data/raw/videos/csv"

In [3]:
# all_channels / all_videos 로드 (앞 단계에서 이어지는 경우 생략 가능)
all_channels = pd.read_csv(os.path.join(DATA_DIR_channels, "all_channels.csv"))
all_videos   = pd.read_csv(os.path.join(DATA_DIR_videos, "all_videos.csv"))
all_videos["published_at"] = pd.to_datetime(all_videos["published_at"], utc=True, errors="coerce")
all_videos["is_shorts"]    = all_videos["is_shorts"].astype(str).str.lower() == "true"


/var/folders/fd/8k73lf_11pjfmg3f69043_q00000gn/T/ipykernel_70643/3626409284.py:3: DtypeWarning: Columns (0,7) have mixed types. Specify dtype option on import or set low_memory=False.
  all_videos   = pd.read_csv(os.path.join(DATA_DIR_videos, "all_videos.csv"))


In [4]:
# videos → 채널 단위 집계
NOW = pd.Timestamp.now(tz="UTC")

def aggregate_videos(group):
    dates    = group["published_at"].sort_values(ascending=False).tolist()
    intervals = [(dates[i] - dates[i+1]).days for i in range(len(dates) - 1)] # 업로드 주기 계산

    t3 = NOW - pd.Timedelta(days=90) # 3달 기준
    t6 = NOW - pd.Timedelta(days=180) # 6달 기준
    recent = sum(1 for d in dates if d >= t3)
    prev   = sum(1 for d in dates if t6 <= d < t3)

    shorts = group[group["is_shorts"] == True]
    normal = group[group["is_shorts"] == False]
    merged = group[["view_count","like_count","comment_count"]].dropna()

    return pd.Series({
        # 활동성
        "collected_video_count"     : len(dates),
        "days_since_last_upload"    : (NOW - dates[0]).days,
        "avg_upload_interval_days"  : round(float(__import__("numpy").mean(intervals)), 2)  if intervals else float("nan"),
        "std_upload_interval_days"  : round(float(__import__("numpy").std(intervals)), 2)   if intervals else float("nan"),
        "max_gap_days"              : max(intervals)                                         if intervals else float("nan"),
        "hiatus_count_30d"          : sum(1 for d in intervals if d >= 30),
        "upload_freq_change_rate"   : round((recent - prev) / (prev + 1e-9), 4),
        # 성과
        "avg_view_count"            : round(group["view_count"].mean(), 2),
        "std_view_count"            : round(group["view_count"].std(), 2),
        "avg_like_count"            : round(group["like_count"].mean(), 2),
        "avg_comment_count"         : round(group["comment_count"].mean(), 2),
        "avg_engagement_rate"       : round(
            ((merged["like_count"] + merged["comment_count"]) / (merged["view_count"] + 1e-9)).mean(), 6
        ) if not merged.empty else float("nan"),
        # Shorts
        "shorts_ratio"              : round(len(shorts) / (len(group) + 1e-9), 4),
        "avg_shorts_view"           : round(shorts["view_count"].mean(), 2) if not shorts.empty else float("nan"),
        "avg_normal_view"           : round(normal["view_count"].mean(), 2) if not normal.empty else float("nan"),
    })

video_agg = all_videos.groupby("channel_id").apply(aggregate_videos, include_groups=False).reset_index()
print(f"집계 후 shape : {video_agg.shape}")
video_agg.head(3)


집계 후 shape : (8092, 16)


,channel_id,collected_video_count,days_since_last_upload,avg_upload_interval_days,std_upload_interval_days,max_gap_days,hiatus_count_30d,upload_freq_change_rate,avg_view_count,std_view_count,avg_like_count,avg_comment_count,avg_engagement_rate,shorts_ratio,avg_shorts_view,avg_normal_view
0,2024-06-14 10:00:44+00:00,1.0,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN
1,2024-06-16 10:00:29+00:00,1.0,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN
2,2024-06-17 10:00:47+00:00,1.0,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN


In [5]:
# video_count 컬럼명 충돌 방지
all_channels = all_channels.rename(columns={"video_count": "total_video_count"})

# inner join
df_wide = all_channels.merge(video_agg, on="channel_id", how="inner")

# 불필요 컬럼 제거
df_wide = df_wide.drop(columns=["uploads_playlist", "country"], errors="ignore")

print(f"조인 전 channels : {len(all_channels)}행")
print(f"조인 후 wide     : {len(df_wide)}행  ← 차이 있으면 영상 없는 채널 존재")
print(f"최종 shape       : {df_wide.shape}")
df_wide.head(3)


조인 전 channels : 8108행
조인 후 wide     : 8083행  ← 차이 있으면 영상 없는 채널 존재
최종 shape       : (8083, 22)


,channel_id,title,published_at,subscriber_count,view_count,total_video_count,source_file,collected_video_count,days_since_last_upload,avg_upload_interval_days,...,hiatus_count_30d,upload_freq_change_rate,avg_view_count,std_view_count,avg_like_count,avg_comment_count,avg_engagement_rate,shorts_ratio,avg_shorts_view,avg_normal_view
0,UC8EiJ1MgI0S0UAqmWaTcoBw,기준TV,2018-12-18T10:07:33Z,3,495,17,channels_must_0_49.csv,17.0,2400.0,18.88,...,4.0,0.0,29.12,30.87,0.41,0.06,0.026937,0.00,NaN,29.12
1,UCTdCqgNr9RFJWXK8qZgxL5g,James1004,2008-03-17T12:33:16Z,14,6,2,channels_must_0_49.csv,4.0,1150.0,499.33,...,1.0,0.0,1.50,2.38,0.00,0.00,0.000000,0.00,NaN,1.50
2,UCMCcdMTgIzzdpa2iP8RtS8g,ted youngtae Noh,2011-11-11T01:21:17Z,11,280,4,channels_must_0_49.csv,4.0,1621.0,118.33,...,1.0,0.0,70.00,88.18,0.75,0.25,0.007338,0.25,9.0,90.33


In [7]:
# null 확인
null_w = df_wide.isnull().sum()
print("=== null 현황 ===")
print(null_w[null_w > 0] if null_w[null_w > 0].any() else "없음")


=== null 현황 ===
avg_upload_interval_days      53
std_upload_interval_days      53
max_gap_days                  51
std_view_count                51
avg_like_count               174
avg_comment_count             84
avg_engagement_rate          246
avg_shorts_view             1702
avg_normal_view              172
dtype: int64


In [8]:
DATA_DIR = "/Users/sj.kang/Desktop/project2/SKN30-2nd-1Team/data/raw"
out_wide = os.path.join(DATA_DIR, "dataset_wide.csv")
df_wide.to_csv(out_wide, index=False, encoding="utf-8-sig")
print(f"저장 완료 : {out_wide}")
print(f"shape     : {df_wide.shape}")


저장 완료 : /Users/sj.kang/Desktop/project2/SKN30-2nd-1Team/data/raw/dataset_wide.csv
shape     : (8083, 22)


### 3-2. Long 조인 (영상 단위, LSTM·시계열용)

각 영상 행에 채널 기본 정보(구독자 수, 생성일 등)를 붙입니다.


In [9]:
# channels에서 영상별로 붙일 컬럼만 선택
ch_meta = all_channels[[
    "channel_id", "title", "published_at", "country",
    "subscriber_count", "view_count", "total_video_count"
]].rename(columns={
    "published_at" : "channel_created_at",
    "view_count"   : "channel_total_views",
})

df_long = all_videos.merge(ch_meta, on="channel_id", how="left")

print(f"조인 후 long shape : {df_long.shape}")
print(f"채널 정보 없는 영상 : {df_long['title'].isnull().sum()}개")
df_long.head(3)


조인 후 long shape : (376988, 18)
채널 정보 없는 영상 : 9개


,idx,channel_id,channel_title,video_id,video_title,published_at,duration_sec,is_shorts,view_count,like_count,comment_count,source_file,title,channel_created_at,country,subscriber_count,channel_total_views,total_video_count
0,0,UCo3Yj54VtkEvQX9cLHKklzw,장정숙,eM25H_bOkJ4,171024 [장정숙 의원] 2017 국정감사_국립대 신입간호사 임금착취 문제 해결,2018-05-28 02:16:23+00:00,66.0,False,75.0,0.0,0.0,videos_0000-0049.csv,장정숙,2016-06-08T06:54:31Z,NaN,1.0,493.0,3.0
1,0,UCo3Yj54VtkEvQX9cLHKklzw,장정숙,KoovsJOVOEg,[국회의원 장정숙] 전반기 교육문화체육관광위원회 활동,2018-07-03 04:16:52+00:00,224.0,False,226.0,10.0,0.0,videos_0000-0049.csv,장정숙,2016-06-08T06:54:31Z,NaN,1.0,493.0,3.0
2,0,UCo3Yj54VtkEvQX9cLHKklzw,장정숙,VhXviHrA_yw,"191211 언제나 국민의편, 국회의원 장정숙",2019-12-11 05:58:53+00:00,86.0,False,192.0,2.0,0.0,videos_0000-0049.csv,장정숙,2016-06-08T06:54:31Z,NaN,1.0,493.0,3.0


In [10]:
out_long = os.path.join(DATA_DIR, "dataset_long.csv")
df_long.to_csv(out_long, index=False, encoding="utf-8-sig")
print(f"저장 완료 : {out_long}")
print(f"shape     : {df_long.shape}")


저장 완료 : /Users/sj.kang/Desktop/project2/SKN30-2nd-1Team/data/raw/dataset_long.csv
shape     : (376988, 18)


In [11]:
df = pd.read_csv("/Users/sj.kang/Desktop/project2/SKN30-2nd-1Team/data/raw/dataset_long.csv")
df.head()

/var/folders/fd/8k73lf_11pjfmg3f69043_q00000gn/T/ipykernel_70643/3507194148.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/Users/sj.kang/Desktop/project2/SKN30-2nd-1Team/data/raw/dataset_long.csv")


,idx,channel_id,channel_title,video_id,video_title,published_at,duration_sec,is_shorts,view_count,like_count,comment_count,source_file,title,channel_created_at,country,subscriber_count,channel_total_views,total_video_count
0,0,UCo3Yj54VtkEvQX9cLHKklzw,장정숙,eM25H_bOkJ4,171024 [장정숙 의원] 2017 국정감사_국립대 신입간호사 임금착취 문제 해결,2018-05-28 02:16:23+00:00,66.0,False,75.0,0.0,0.0,videos_0000-0049.csv,장정숙,2016-06-08T06:54:31Z,NaN,1.0,493.0,3.0
1,0,UCo3Yj54VtkEvQX9cLHKklzw,장정숙,KoovsJOVOEg,[국회의원 장정숙] 전반기 교육문화체육관광위원회 활동,2018-07-03 04:16:52+00:00,224.0,False,226.0,10.0,0.0,videos_0000-0049.csv,장정숙,2016-06-08T06:54:31Z,NaN,1.0,493.0,3.0
2,0,UCo3Yj54VtkEvQX9cLHKklzw,장정숙,VhXviHrA_yw,"191211 언제나 국민의편, 국회의원 장정숙",2019-12-11 05:58:53+00:00,86.0,False,192.0,2.0,0.0,videos_0000-0049.csv,장정숙,2016-06-08T06:54:31Z,NaN,1.0,493.0,3.0
3,1,UCyABUa7lzjsV4o5PfSFqymw,심해 생존일지스타트의,M7vYRNXEkwk,"여기서 만큼은 심해가 아니다!중딩 스타트의 플레 루시우볼!(feat,뽈쟁이,버틀너버",2017-08-19 12:12:26+00:00,270.0,False,277.0,2.0,5.0,videos_0000-0049.csv,심해 생존일지스타트의,2017-08-06T17:58:34Z,NaN,1.0,942.0,3.0
4,1,UCyABUa7lzjsV4o5PfSFqymw,심해 생존일지스타트의,XwcIz07B_RM,[start]:심해 매드무비-둠피스트,2017-10-09 06:47:10+00:00,344.0,False,312.0,0.0,1.0,videos_0000-0049.csv,심해 생존일지스타트의,2017-08-06T17:58:34Z,NaN,1.0,942.0,3.0
